# 0. Libraries

In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import rl_mind.envs
from dataclasses import dataclass, replace
from scipy import stats
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.core import Action, Actor
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import EpisodeCollector, TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings

from agent import *
from loss import *
from config import *
from bias import *
from train import *
from results import load_runs 
from results import load_runs

# Benchmark

`run_ddpg` and `run_td3` are defined in `train.py`. Each run is executed in its own process (`run_job`),
and saves its results in `outputs/<algo>/<variant>/S<seed>-<date>/results.json`.

A variant overrides fields of the default configuration (`config.py`): do not change the defaults to create a variant.

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

silence_known_warnings()
seeds = [1, 2, 3, 4, 5]

# Each variant overrides fields of the default config (config.py); its runs are named after it
deep = {"actor_hidden": (256, 256, 256), "critic_hidden": (256, 256, 256)}
variants = [
    {},                       # base : 2 couches cachées
    {"layer_norm": True},     # LayerNorm dans le(s) critique(s)
    deep,                     # 3 couches cachées (acteur et critique)
    {"utd_ratio": 4},         # 4 pas de gradient par transition
]
jobs = [(algo, seed, overrides) for overrides in variants for algo in ("ddpg", "td3") for seed in seeds]

# One process per run (at most 5 at once: memory); progress is visible in tensorboard (`outputs/`)
with ProcessPoolExecutor(max_workers=5, mp_context=mp.get_context("spawn")) as pool:
    list(pool.map(run_job, jobs))

# Analysis

`load_runs()` reads the finished runs from `outputs/`: these cells can be run while the benchmark is still in progress.
The runs of the previous protocol (separate Monte Carlo episodes) are in `archive/separate-mc-300k/`.

In [ ]:
import numpy as np


def reward_curves(runs) -> tuple[list[int], np.ndarray]:
    """Mean evaluation reward of each run: steps, array `[n_seeds, n_evals]`"""
    steps = [result.step for result in runs[0][0]]
    rewards = np.array([[result.mean for result in history] for history, _ in runs])
    return steps, rewards


def bias_curves(runs, key: str = "bias") -> tuple[list[int], np.ndarray]:
    """A metric of the evaluations of each run (NaN when it could not be computed)"""
    steps = [step for step, _ in runs[0][1]]
    values = np.array([[bias.get(key, np.nan) for _, bias in history] for _, history in runs])
    return steps, values


def plot_curves(ylabel: str, **curves):
    """Mean +/- std across seeds"""
    for name, (steps, values) in curves.items():
        mean, std = np.nanmean(values, axis=0), np.nanstd(values, axis=0)
        (line,) = plt.plot(steps, mean, label=name)
        plt.fill_between(steps, mean - std, mean + std, alpha=0.2, color=line.get_color())
    plt.xlabel("steps")
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()


def last_evals(runs, key: str | None = None) -> np.ndarray:
    """One value per seed: the reward (or a metric) averaged over the last 5 evaluations"""
    _, values = reward_curves(runs) if key is None else bias_curves(runs, key)
    return np.nanmean(values[:, -5:], axis=1)

In [ ]:
runs = load_runs()
print({name: len(r) for name, r in runs.items()})

# One figure per algorithm and metric, with all its variants
for algo in ("ddpg", "td3"):
    algo_runs = {name: r for name, r in runs.items() if name.startswith(algo)}
    if not algo_runs:
        continue
    plot_curves(f"{algo}: cumulated reward", **{n: reward_curves(r) for n, r in algo_runs.items()})
    for key in ("bias", "normalized_bias", "q", "mc_return", "episode_length", "truncated"):
        plot_curves(f"{algo}: {key}", **{n: bias_curves(r, key) for n, r in algo_runs.items()})

In [ ]:
# Welch t-tests (one value per seed): each variant vs the base of its algorithm, and DDPG vs TD3
for name, r in runs.items():
    algo, variant = name.split("/", 1)
    if variant == "base" or f"{algo}/base" not in runs:
        continue
    for label, key in (("reward", None), ("bias", "bias")):
        base, other = last_evals(runs[f"{algo}/base"], key), last_evals(r, key)
        p = stats.ttest_ind(base, other, equal_var=False).pvalue
        print(f"{name:70s} {label:6s}: base {base.mean():7.1f} -> {other.mean():7.1f}  (p = {p:.3f})")

if "ddpg/base" in runs and "td3/base" in runs:
    for label, key in (("reward", None), ("bias", "bias")):
        ddpg, td3 = last_evals(runs["ddpg/base"], key), last_evals(runs["td3/base"], key)
        p = stats.ttest_ind(ddpg, td3, equal_var=False).pvalue
        print(f"DDPG vs TD3 (base) {label:6s}: {ddpg.mean():7.1f} vs {td3.mean():7.1f}  (p = {p:.3f})")